In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# IMPORTS
import numpy as np
import os
import shutil
from itertools import chain
import matplotlib.pyplot as plt
import wave
import re
import pprint

In [3]:
# GLOBAL VARIABLES
TITLE_LIST = ['LEARNING ABOUT SOMETHING','THINGS START TO GET QUITE DEEP','ABSOLUTE EXISTENTIAL CRISIS','TRANSITION TO HOPE',"EXISTENCE'S GREATNESS",'THERE IS NO END, ONLY NEW BEGINNINGS','FREEDOM PARK']
TRACK_COUNT_LIST = [2,3,2,2,3,1,1]
VIP_TITLE_LIST = ['ABSOLUTE EXISTENTIAL CRISIS','FREEDOM PARK']
ORDER_FILE_NAME = "order.txt"
MUSIC_FOLDER_NAME = 'Tracks'
MUSIC_FILE_FORMAT = 'wav' # DO NOT INCLUDE '.'
INERTIA = 0.01
PLAYLIST_FOLDER_NAME = 'Random playlist'
HISTORY_FILE_NAME = 'anti-staleness.txt'
HISTORY_FILE_SEPARATOR = '======== EXECUTION SEPARATOR ========'
HISTORY_LIMIT = 3

In [4]:
# Read classification from order file
def get_order():
    # Read file
    with open(ORDER_FILE_NAME, 'r') as order_file:
        order_str = order_file.read()
    order_dict = {}
    dash_line = '-------------------------------------------'
    # Retrieve each song list per category
    for title in TITLE_LIST:
        # Case for last category
        if title == TITLE_LIST[-1]:
            if 'Childhood' in order_str[order_str.find(title):]:
                order_dict[title] = ['Childhood']
        else:
            # Title with dashes
            fancy_title = dash_line + '\n' + title + '\n' + dash_line + '\n'
            # Index of first song of the category
            list_start = order_str.find(fancy_title) + len(fancy_title)
            # Index of last song of the category
            list_end = order_str.find('\n----',list_start)
            # Add song list for the category
            order_dict[title] = order_str[list_start:list_end].split('\n')
    return order_dict

In [5]:
# TESTING

def get_order():
    # Read file
    with open(ORDER_FILE_NAME, 'r') as order_file:
        order_str = order_file.read()
    order_dict = {}
    # Retrieve each song list per category
    title_list = re.findall(r'-{2,}\n(.*?)\n-{2,}', order_str, re.DOTALL)
    for title in title_list:
        # Index of title
        title_start = order_str.find(title)
        # Index of first song of the category
        list_start = order_str.find('--\n', title_start) + 3
        # Index of last song of the category
        list_end = order_str.find('\n----',list_start)
        # Add song list for the category
        if list_end == -1:
            order_dict[title] = order_str[list_start:].split('\n')
        else:
            order_dict[title] = order_str[list_start:list_end].split('\n')
    return order_dict

In [6]:
print(get_order())

{'LEARNING ABOUT SOMETHING': ['Spare Signal', 'La dorme', 'Fading images', 'Horizon', 'Cross Purpose Turf War', 'Sargasso Scene', '6 doors in', 'Calle elisif', 'Domes', 'A night in the snow', 'The sprawling technopoly', 'A strange interlude', 'Goats meadow', 'In the space between shades', 'Final flight of the space rangers', 'Secret geometric states', 'Passing the Nebula (when the attack began)', 'Unmarked Exit', 'Haze', 'Batman kill Spiderman', 'Buttered stairways', 'Porphry', 'Bastard computer', 'Cemented airspace', 'Mountaintop'], 'THINGS START TO GET QUITE DEEP': ['A different kind of morning', 'Zam clock watching', 'Trailblazer', 'Shameful lakes', 'Entering Antares', 'Seagull meets Treeline, February 3 2013', 'The road to Mars', 'A hydrogen molecule', 'Absolute Stillness', 'On a spinning disk, dragons fly', 'Insects', 'Rain Sketch', 'In the Bowels of Flight', 'What we found in the trap', 'Transmetric', 'Entering Mercurys airspace', 'Light Speed Queens', 'Crowed Cloud', 'Time is ma

In [7]:
# TESTING

import re

# El texto de ejemplo
texto = """
New dawn form
Past the house of silence
-------------------------------------------
ABSOLUTE EXISTENTIAL CRISIS
-------------------------------------------
Entering the Temple of the Sun Sphere
Numbers Farm
-------------------------------------------
TRANSITION TO HOPE
-------------------------------------------
Spaceum 78
Naxxil
Cytosyoan
"""

# Patrón de expresión regular para encontrar las substrings entre separadores '-'
patron = r'-{2,}\n(.*?)\n-{2,}'

# Buscar todas las coincidencias usando re.findall
coincidencias = re.findall(patron, texto, re.DOTALL)

# Imprimir las substrings encontradas
for match in coincidencias:
    print(match.strip())  # .strip() para eliminar espacios en blanco adicionales



ABSOLUTE EXISTENTIAL CRISIS
TRANSITION TO HOPE


In [8]:
# Read all music files
def get_music_files():
    file_list = os.listdir(MUSIC_FOLDER_NAME)
    return [file.replace('.' + MUSIC_FILE_FORMAT, '') for file in file_list if '.' + MUSIC_FILE_FORMAT.lower() in file.lower()]

In [9]:
# Find all music tracks not included in order_dict
def find_unordered_missing_tracks(order_dict, music_files_list):
    # List of all tracks in order_dict
    ordered_music_list = list(chain(*order_dict.values()))
    # Not ordered tracks
    not_ordered_list = [track for track in music_files_list if track not in ordered_music_list]
    # Missing tracks
    missing_list = [track for track in ordered_music_list if track not in music_files_list]
    return not_ordered_list, missing_list

In [10]:
# Read and classify music files
def get_music_files_classified():
    # Read classification from order file
    order_dict = get_order()
    # Read all music files
    music_files_list = get_music_files()
    # Find all music tracks not included in order_dict and missing files
    print('Finding missing tracks...')
    not_ordered_list, missing_list = find_unordered_missing_tracks(order_dict, music_files_list)
    if not_ordered_list:
        # Print unordered tracks
        print('Found tracks NOT included in {}'.format(ORDER_FILE_NAME))
        for track in not_ordered_list:
            print('\t· {}'.format(track))
    else:
        print('All tracks are included in {}'.format(ORDER_FILE_NAME))
    if missing_list:
        # Print missing tracks
        print('Ordered tracks NOT found in {} folder'.format(MUSIC_FOLDER_NAME))
        for track in missing_list:
            print('\t· {}'.format(track))
            # Exclude missing track from order dict
            for key in order_dict.keys():
                if track in order_dict[key]:
                    order_dict[key].remove(track)
                    break
    else:
        print('All ordered tracks are included in {} folder'.format(MUSIC_FOLDER_NAME))
    # Read track history from file
    print('Removing tracks selected in the past...')
    with open(HISTORY_FILE_NAME, 'r') as file:
        history_list = [track.replace('\n', '') for track in file.readlines()]
    if len(history_list) > 0:
        # Remove all classified tracks contained in history
        removal_counter = 0
        for key in order_dict:
            pre_removal_len = len(order_dict[key])
            order_dict[key] = [track for track in order_dict[key] if track not in history_list]
            removal_counter += pre_removal_len - len(order_dict[key])
        print('Successfully removed {} tracks'.format(removal_counter))
    else:
        print('Zero tracks found in {} history file'.format(HISTORY_FILE_NAME))
    return order_dict

In [11]:
def get_average_track_length(track_list):
    # Initialize track length dict
    track_lens = {}
    # Iterate for each track
    for track in track_list:
        # Open track file using wave lib
        with wave.open(os.path.join(MUSIC_FOLDER_NAME + '/', track + '.' + MUSIC_FILE_FORMAT), 'rb') as track_file:
            # Add track length
            track_lens[track] = float(track_file.getnframes()) / track_file.getframerate()
    # Calculate average length
    average_len = sum(track_lens.values()) / len(track_lens)
    return average_len, track_lens

In [12]:
def generate_prob_dist(n, top, inertia):
    """
    Genera una distribución de probabilidad con n elementos,
    donde la posición con la probabilidad más alta está en 'top'.
    La probabilidad de cada elemento dependerá de la distancia
    entre su posición y la posición marcada por 'top'.
    El 'decay_rate' controla la velocidad a la que la probabilidad
    disminuye o aumenta según se aleja o se acerca a 'top',
    mientras que 'inertia' controla cómo cambia el 'decay_rate'
    según se aleja o se acerca a 'top'.
    """

    # Crea un vector con la posición de cada elemento en la distribución
    positions = np.arange(n)

    # Calcula la distancia entre cada posición y la posición marcada por 'top'
    distances = np.abs(positions - top * n)

    # Calcula el decay_rate en función de la distancia y de la inertia
    decay_rates = 1 - inertia + inertia * (1 - distances / (n / 2))
    decay_rates = np.clip(decay_rates, 0, 1)

    # Calcula las probabilidades de cada elemento de la distribución
    prob_dist = decay_rates ** distances

    # Normaliza las probabilidades para que sumen 1
    prob_dist = prob_dist / np.sum(prob_dist)

    return prob_dist

In [13]:
# Choose a random element from list
def random_choose(top, inertia, elements):
    prob_dist = generate_prob_dist(len(elements), top, inertia)
    chosen_element = np.random.choice(elements, p=prob_dist)
    return chosen_element

In [14]:
# Choose random tracks froma list
def choose_multiple_tracks(track_count, track_list):
    # Calculate top values
    if track_count == 1:
        top_list = [0.5]
    elif track_count == 2:
        top_list = [1/3, 2/3]
    elif track_count == 3:
        top_list = [0, 0.5 ,1]
    else:
        top_list = []
        top_step = 1 / (track_count + 1)
        top_acum = top_step
        for _ in range(track_count):
            top_list.append(top_acum)
            top_acum += top_step
    # Get average track length and track lengths
    average_len, track_lens = get_average_track_length(track_list)
    # Random choose for each top value
    chosen_list = []
    top_index = 0
    total_len = 0
    len_limit = average_len * track_count
    loop_counter = 0
    new_history_str = ''
    while total_len < len_limit:
        loop_counter += 1
        # Choose track randomly
        chosen_track = random_choose(top_list[top_index], INERTIA, track_list)
        print('\t\tTrack {} chosen: {}'.format(loop_counter, chosen_track))
        # Append to chosen list
        chosen_list.append(chosen_track)
        # Remove from track list, so it's not chosen twice
        track_list.remove(chosen_track)
        # Add chosen track to history
        new_history_str += chosen_track + '\n'
        # Update total length
        total_len += track_lens[chosen_track]
        # If total length is bypassing the average
        if total_len > average_len * (top_index + 1):
            # Update top index
            top_index += 1
    return chosen_list, new_history_str[:-1]

In [15]:
def update_history_file(total_new_history_str):
    print('Updating {} history file...'.format(HISTORY_FILE_NAME))
    # Read history file
    with open(HISTORY_FILE_NAME, 'r+') as file:
        total_old_history_str = file.read()
        # Add new tracks below
        total_new_history_str = total_old_history_str + '\n' + total_new_history_str
        # Count number of separators
        sep_count = total_new_history_str.count(HISTORY_FILE_SEPARATOR)
        # If history limit has been exceeded
        if sep_count > HISTORY_LIMIT:
            # Remove tracks from first separator
            first_separator_end_index = total_new_history_str.find(HISTORY_FILE_SEPARATOR, len(HISTORY_FILE_SEPARATOR))
            total_new_history_str = total_new_history_str[first_separator_end_index:]
        # Write new content
        file.seek(0) # Move pointer to the beginning to overwrite all
        file.write(total_new_history_str)
        file.truncate() # If new content is shorter than the previous one, removes exceeding content
    print('History updated successfully')

In [16]:
# Generate random playlist
def generate_random_playlist(music_files_dict):
    print('Generating random playlist...')
    random_playlist = []
    track_count_dict = dict(zip(TITLE_LIST,TRACK_COUNT_LIST))
    total_new_history_str = HISTORY_FILE_SEPARATOR
    for key in music_files_dict.keys():
        # Get number of tracks from this section to include in the playlist
        track_count = track_count_dict[key]
        # Case for vip titles
        if key in VIP_TITLE_LIST:
            print('\tChoosing {} tracks from {} section...'.format(track_count, key))
            random_playlist += music_files_dict[key]
            for i, track in enumerate(music_files_dict[key]):
                print('\t\tTrack {} chosen: {}'.format(i+1, track))
        else:
            # Choose random tracks for this section
            print('\tChoosing around {} tracks from {} section...'.format(track_count, key))
            chosen_list, new_history_str = choose_multiple_tracks(track_count, music_files_dict[key])
            random_playlist += chosen_list
            total_new_history_str += '\n' + new_history_str
    # Update history file
    update_history_file(total_new_history_str)
    return random_playlist

In [17]:
# Create file copies from list
def create_files_from_list(playlist):
    print('Creating file copies...')
    # Check if there are files in the folder
    file_list = os.listdir(PLAYLIST_FOLDER_NAME)
    if file_list:
        print('\tDeleting existing playlist...')
        # Delete files
        for intruder_file in file_list:
            os.remove(PLAYLIST_FOLDER_NAME + '/' + intruder_file)
        print('\tPlaylist successfully deleted')
    # Create an enumerated copy for each track from the playlist
    output_track_count = 0
    for track in playlist:
        output_track_count += 1
        print('\tAdding Track' + str(output_track_count).zfill(2) + '...')
        file_src = MUSIC_FOLDER_NAME + '/' + track + '.' + MUSIC_FILE_FORMAT
        dst_folder = PLAYLIST_FOLDER_NAME + '/Track' + str(output_track_count).zfill(2) + '.' + MUSIC_FILE_FORMAT
        shutil.copy(file_src, dst_folder)

In [18]:
# Main process
def generate_random_playlist_as_files():
    print('generate_random_playlist_as_files() iniciated')
    # Read and classify music files
    music_files_dict = get_music_files_classified()
    # Generate random playlist
    random_playlist = generate_random_playlist(music_files_dict)
    # Create file copies
    create_files_from_list(random_playlist)
    print('generate_random_playlist_as_files() successfully executed')
    print('\n ~~~~~~~~~~ eNjOy YoUr OdYsSeY ~~~~~~~~~~')

In [19]:
generate_random_playlist_as_files()

generate_random_playlist_as_files() iniciated
Finding missing tracks...
All tracks are included in order.txt
All ordered tracks are included in Tracks folder
Removing tracks selected in the past...
Successfully removed 46 tracks
Generating random playlist...
	Choosing around 2 tracks from LEARNING ABOUT SOMETHING section...
		Track 1 chosen: Buttered stairways
		Track 2 chosen: Calle elisif
		Track 3 chosen: Final flight of the space rangers
	Choosing around 3 tracks from THINGS START TO GET QUITE DEEP section...
		Track 1 chosen: Light Speed Queens
		Track 2 chosen: Time is magic
	Choosing 2 tracks from ABSOLUTE EXISTENTIAL CRISIS section...
		Track 1 chosen: Entering the Temple of the Sun Sphere
		Track 2 chosen: Numbers Farm
	Choosing around 2 tracks from TRANSITION TO HOPE section...
		Track 1 chosen: Peace
		Track 2 chosen: Galactic shadow goverment
		Track 3 chosen: Flight over calm seas
	Choosing around 3 tracks from EXISTENCE'S GREATNESS section...
		Track 1 chosen: Back to the